In [1]:
# ==============================================================================
# 3rd Sem AEC - Experiment 1: Exploratory Data Analysis (EDA)
# Datasets: MNIST (Grayscale), Fashion-MNIST (Grayscale), CIFAR-10 (Color RGB)
# Output: Saves .png figures in ./eda_outputs/ and creates ./eda_outputs.zip
# ==============================================================================

import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist, fashion_mnist, cifar10

# 1. Setup output directory
OUT_DIR = "eda_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 150

# Class labels for human-readable plots
LABELS = {
    "MNIST": [str(i) for i in range(10)],
    "Fashion-MNIST": [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ],
    "CIFAR-10": [
        "Airplane", "Automobile", "Bird", "Cat", "Deer",
        "Dog", "Frog", "Horse", "Ship", "Truck"
    ],
}

# 2. Load datasets
datasets = {
    "MNIST": mnist.load_data(),
    "Fashion-MNIST": fashion_mnist.load_data(),
    "CIFAR-10": cifar10.load_data(),
}

# 3. Print dataset summary & pixel statistics
print("=" * 75)
print("DATASET SUMMARY & PIXEL STATISTICS")
print("=" * 75)

summary_stats = {}
for name, ((x_train, y_train), (x_test, y_test)) in datasets.items():
    y_train_flat = y_train.ravel()
    y_test_flat = y_test.ravel()

    # Ensure channel dimension is clear for shape reporting
    train_shape = x_train.shape
    test_shape = x_test.shape

    p_min = float(x_train.min())
    p_max = float(x_train.max())
    p_mean = float(x_train.mean())
    p_std = float(x_train.std())
    num_zeros = int((x_train == 0).sum())
    zero_pct = 100.0 * num_zeros / x_train.size

    summary_stats[name] = {
        "train_n": len(x_train),
        "test_n": len(x_test),
        "shape": train_shape[1:],
        "mean": p_mean,
        "std": p_std,
        "zero_pct": zero_pct,
        "y_train": y_train_flat,
        "y_test": y_test_flat,
    }

    print(f"\n[{name}]")
    print(f"  Train set : {x_train.shape} | Labels: {y_train_flat.shape}")
    print(f"  Test set  : {x_test.shape}  | Labels: {y_test_flat.shape}")
    print(f"  Dtype     : {x_train.dtype} | Range: [{p_min:.0f}, {p_max:.0f}]")
    print(f"  Train Mean: {p_mean:.2f} | Std Dev: {p_std:.2f} | Zero Pixels: {zero_pct:.2f}%")

# 4. Save sample image grids (.png) for each dataset
for name, ((x_train, y_train), _) in datasets.items():
    y_flat = y_train.ravel()
    class_names = LABELS[name]
    is_rgb = (x_train.ndim == 4 and x_train.shape[-1] == 3)

    fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
    fig.suptitle(f"{name} - Sample Training Images (1 per Class)", fontsize=13, fontweight="bold")

    for cls_idx in range(10):
        ax = axes[cls_idx // 5, cls_idx % 5]
        idx = np.where(y_flat == cls_idx)[0][0]
        img = x_train[idx]

        if is_rgb:
            ax.imshow(img)
        else:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)

        ax.set_title(f"{cls_idx}: {class_names[cls_idx]}", fontsize=9)
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    save_path = os.path.join(OUT_DIR, f"{name.lower().replace('-', '_')}_samples.png")
    plt.savefig(save_path)
    plt.close(fig)
    print(f"Saved: {save_path}")

# 5. Save class distribution bar charts (.png) for Train & Test splits
for name, info in summary_stats.items():
    class_names = LABELS[name]
    x_pos = np.arange(10)
    width = 0.4

    train_counts = np.bincount(info["y_train"], minlength=10)
    test_counts = np.bincount(info["y_test"], minlength=10)

    fig, ax = plt.subplots(figsize=(11, 4.8))
    ax.bar(x_pos - width / 2, train_counts, width, label="Train Split")
    ax.bar(x_pos + width / 2, test_counts, width, label="Test Split")

    ax.set_title(f"{name} - Class Distribution Across Train & Test Sets", fontsize=12, fontweight="bold")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Samples")
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{i}\n{class_names[i]}" for i in range(10)], fontsize=8, rotation=20, ha="right")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    save_path = os.path.join(OUT_DIR, f"{name.lower().replace('-', '_')}_class_dist.png")
    plt.savefig(save_path)
    plt.close(fig)
    print(f"Saved: {save_path}")

# 6. Save pixel intensity distribution comparison (.png)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle("Training Set Pixel Intensity Distributions (0 - 255)", fontsize=13, fontweight="bold")

for ax, (name, ((x_train, _), _)) in zip(axes, datasets.items()):
    # Subsample up to 2,000 images for fast histogram plotting in Colab
    sample_x = x_train[:2000].ravel()
    ax.hist(sample_x, bins=32, density=True, alpha=0.85)
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Density")
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.92])
hist_path = os.path.join(OUT_DIR, "pixel_intensity_histograms.png")
plt.savefig(hist_path)
plt.close(fig)
print(f"Saved: {hist_path}")

# 7. Create a ZIP archive of all .png outputs for easy download from Colab
zip_base = "eda_outputs"
zip_file = shutil.make_archive(zip_base, "zip", OUT_DIR)
print(f"\nAll PNG files archived to: {zip_file}")
print("Files in folder:", sorted(os.listdir(OUT_DIR)))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 441s 3us/step
DATASET SUMMARY & PIXEL STATISTICS

[MNIST]
  Train set : (60000, 28, 28) | Labels: (60000,)
  Test set  : (10000, 28, 28)  | Labels: (10000,)
  Dtype     : uint8 | Range: [0, 255]
  Train Mean: 33.32 | Std Dev: 78.57 | Zero Pixels: 80.88%

[Fashion-MNIST]
  Train set : (60000, 28, 28) | Labels: (60000,)
  Test set  : (10000, 28, 28)  | Labels: (10000,)
  Dtype     : uint8 | Range: [0, 255]
  Train Mean: 72.94 | Std Dev: 90.02 | Zero Pixels: 50.21%

[CIFAR-10]
  Train set : (50000, 32, 32, 3) | Labels: (50000,)
  Test set  : (10000, 32, 32, 3)  | Labels: (10000,)
  Dtype     : uint8 | Range: [0, 255]
  Train Mean: 120.71 | Std Dev: 64.15 | Zero Pixels: 0.24%
Saved: eda_outputs/mnist_samples.png
Sa